# Phase 2.5: Embed Detection Models into FLUX

**Objective:** Embed detection/camera models into .flx for offline Memory Fabric.

**Models:**
| Model | Size | Purpose |
|-------|------|--------|
| MiDaS DPT-Large | ~400 MB | Depth estimation |
| InsightFace buffalo_l | ~250 MB | Face detection + recognition |
| OWL-ViT2 | ~400 MB | Open-vocabulary object detection |
| HRNet-W32 | ~120 MB | Pose estimation (17 keypoints) |
| Pyannote | ~100 MB | Speaker diarization |

**Input:** `Flux-Apex-V1.flx` (v7.0 with 7 embedded models)  
**Output:** `Flux-Apex-V1.flx` (v7.1 with 12 embedded models)

---
⚠️ **CRITICAL:** NumPy must be pinned to <2.0 FIRST before installing detection libraries!

In [ ]:
"""Cell 1: Pin NumPy FIRST (before any imports)"""

import os
import sys
from pathlib import Path

# ════════════════════════════════════════════════════════════════════
# STEP 1: Pin numpy BEFORE anything else touches it
# InsightFace, pyannote, timm use numpy C APIs removed in 2.0+
# ════════════════════════════════════════════════════════════════════
print("[1/5] Pinning numpy<2.0 (MUST be first)...")
os.system('pip install -q "numpy<2.0" --force-reinstall')

# Check if we need kernel restart
import numpy as np
print(f"      NumPy in memory: {np.__version__}")

if np.__version__.startswith('2.'):
    print("\n" + "="*60)
    print("  ⚠️  KERNEL RESTART REQUIRED")
    print("="*60)
    print("  NumPy 1.x was installed but 2.x is still in memory.")
    print("  ")
    print("  👉 Click: Runtime → Restart session (or Kernel → Restart)")
    print("  👉 Then re-run this cell - it will work on second run")
    print("="*60)
    # Don't raise - just stop here so user can restart
else:
    print("      ✓ NumPy 1.x confirmed - good to proceed!")

In [ ]:
"""Cell 2: Environment & Dependencies"""

# Environment detection
if Path('/kaggle').exists():
    ROOT = Path('/kaggle/working/FLUX')
    ENVIRONMENT = 'kaggle'
    if not ROOT.exists():
        os.system('git clone https://github.com/Unseengap/FLUX.git /kaggle/working/FLUX')
elif Path('/content').exists():
    ROOT = Path('/content/FLUX')
    ENVIRONMENT = 'colab'
    if not ROOT.exists():
        os.system('git clone https://github.com/Unseengap/FLUX.git /content/FLUX')
else:
    ROOT = Path('/Users/admin/Desktop/flux')
    ENVIRONMENT = 'local'

print(f"Environment: {ENVIRONMENT}")
print(f"Root: {ROOT}")

# ════════════════════════════════════════════════════════════════════
# CRITICAL: Pin numpy in EVERY install to prevent upgrades
# ════════════════════════════════════════════════════════════════════
print("\n[2/5] Installing ML dependencies (keeping numpy<2.0)...")
os.system('pip install -q "numpy<2.0" transformers accelerate timm')

print("\n[3/5] Installing detection libraries (keeping numpy<2.0)...")
os.system('pip install -q "numpy<2.0" onnxruntime insightface')
os.system('pip install -q "numpy<2.0" pyannote.audio')

# Force numpy back to 1.x in case anything upgraded it
print("\n[4/5] Ensuring numpy<2.0...")
os.system('pip install -q "numpy<2.0" --force-reinstall')

print("\n[5/5] Verifying...")
import numpy as np
print(f"      NumPy: {np.__version__}")
if np.__version__.startswith('2.'):
    print("      ❌ STOP - numpy 2.x detected. Restart kernel and re-run from Cell 1")
else:
    import torch
    print(f"      PyTorch: {torch.__version__}")
    print(f"      CUDA: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"      GPU: {torch.cuda.get_device_name()}")
        print(f"      VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("      ✓ Environment ready")

sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

In [ ]:
"""Cell 3: Initialize Logger & Load Model"""

import gc
import torch
from datetime import datetime
from typing import Dict, Any
from huggingface_hub import hf_hub_download

from flux_utils import PhaseLogger, get_device, _resolve_hf_token

log = PhaseLogger(phase=205)
log.separator("Phase 2.5: Detection Model Embedding")

device = get_device()
log.info(f"Device: {device}")

# HuggingFace token (needed for pyannote)
try:
    hf_token = _resolve_hf_token()
    log.success("HF token resolved")
except:
    hf_token = None
    log.warning("No HF token - pyannote may not download")

# Load model
MODEL_PATH = ROOT / 'checkpoints' / 'Flux-Apex-V1.flx'

if not MODEL_PATH.exists():
    log.info("Downloading from HuggingFace...")
    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    hf_hub_download(
        repo_id='UnseenGAP/FLUX',
        filename='checkpoints/Flux-Apex-V1.flx',
        local_dir=str(ROOT),
        token=hf_token,
    )

flux_model = torch.load(str(MODEL_PATH), map_location='cpu', weights_only=False)
log.info(f"Loaded: {flux_model.get('version', 'unknown')}")
log.info(f"Size: {MODEL_PATH.stat().st_size / 1e9:.2f} GB")
log.info(f"Current models: {list(flux_model.get('models', {}).keys())}")

In [ ]:
"""Cell 4: Embedding Utilities"""

def embed_torch_model(model: torch.nn.Module, quantization: str = 'fp16') -> tuple:
    """Extract weights from PyTorch model."""
    weights = {}
    total_params = 0
    
    for name, param in model.named_parameters():
        w = param.detach().cpu()
        if quantization == 'fp16' and w.dtype == torch.float32:
            w = w.half()
        weights[name] = w
        total_params += param.numel()
    
    for name, buf in model.named_buffers():
        if buf is not None:
            b = buf.detach().cpu()
            if quantization == 'fp16' and b.dtype == torch.float32:
                b = b.half()
            weights[f'_buf_{name}'] = b
    
    return weights, total_params


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def get_size(obj) -> int:
    """Get size in bytes."""
    if isinstance(obj, torch.Tensor):
        return obj.numel() * obj.element_size()
    elif isinstance(obj, dict):
        return sum(get_size(v) for v in obj.values())
    elif isinstance(obj, (list, tuple)):
        return sum(get_size(x) for x in obj)
    return 0

print("✓ Utilities ready")

In [ ]:
"""Cell 5: Embed MiDaS (Depth Estimation)"""

log.cell_start("MiDaS")
cleanup()

midas_ok = False

# Try 1: torch.hub
try:
    print("  Trying torch.hub MiDaS...")
    model = torch.hub.load('intel-isl/MiDaS', 'DPT_Large', pretrained=True, trust_repo=True)
    model.eval()
    weights, params = embed_torch_model(model, 'fp16')
    
    flux_model['models']['depth'] = {
        'base_model': 'intel-isl/MiDaS:DPT_Large',
        'quantization': 'fp16',
        'total_params': params,
        'weights': weights,
        'config': {'input_size': 384},
        'role': 'depth_estimation',
        'tasks': ['depth_map', 'distance_estimation'],
        'lazy_load': True,
    }
    
    print(f"  ✓ MiDaS: {params:,} params ({params*2/1e9:.2f} GB)")
    del model
    cleanup()
    midas_ok = True
    log.success(f"MiDaS embedded: {params:,} params")
    
except Exception as e:
    print(f"  ✗ torch.hub failed: {e}")

# Try 2: HuggingFace DPT
if not midas_ok:
    try:
        print("  Trying HuggingFace DPT...")
        from transformers import DPTForDepthEstimation
        
        model = DPTForDepthEstimation.from_pretrained(
            'Intel/dpt-large',
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
        )
        weights, params = embed_torch_model(model, 'fp16')
        
        flux_model['models']['depth'] = {
            'base_model': 'Intel/dpt-large',
            'quantization': 'fp16',
            'total_params': params,
            'weights': weights,
            'role': 'depth_estimation',
            'tasks': ['depth_map', 'distance_estimation'],
            'lazy_load': True,
        }
        
        print(f"  ✓ DPT: {params:,} params ({params*2/1e9:.2f} GB)")
        del model
        cleanup()
        midas_ok = True
        log.success(f"DPT embedded: {params:,} params")
        
    except Exception as e:
        print(f"  ✗ HF DPT failed: {e}")

if not midas_ok:
    log.warning("Depth: placeholder only")
    flux_model['models']['depth'] = {
        'base_model': 'Intel/dpt-large',
        'role': 'depth_estimation',
        'lazy_load': True,
        'weights': None,
    }

log.cell_end("MiDaS", "PASS" if midas_ok else "PARTIAL")

In [ ]:
"""Cell 6: Embed InsightFace (Face Detection)"""

log.cell_start("InsightFace")
cleanup()

face_ok = False

try:
    print("  Loading InsightFace buffalo_l...")
    from insightface.app import FaceAnalysis
    
    app = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])
    app.prepare(ctx_id=-1)
    
    # Store ONNX model bytes directly
    onnx_models = {}
    total_size = 0
    
    for name, model in app.models.items():
        if hasattr(model, 'model_file') and Path(model.model_file).exists():
            with open(model.model_file, 'rb') as f:
                data = f.read()
            onnx_models[name] = {'bytes': data, 'size': len(data)}
            total_size += len(data)
            print(f"    {name}: {len(data)/1e6:.1f} MB")
    
    if total_size > 0:
        flux_model['models']['face'] = {
            'base_model': 'insightface/buffalo_l',
            'format': 'onnx',
            'total_size_bytes': total_size,
            'total_params': total_size // 4,
            'onnx_models': onnx_models,
            'config': {'det_size': (640, 640)},
            'role': 'face_detection_recognition',
            'tasks': ['face_detect', 'face_recognize', 'age_gender'],
            'lazy_load': True,
        }
        print(f"  ✓ InsightFace: {total_size/1e6:.1f} MB")
        face_ok = True
        log.success(f"InsightFace embedded: {total_size/1e6:.1f} MB")
    
    del app
    cleanup()
    
except Exception as e:
    print(f"  ✗ InsightFace failed: {e}")

if not face_ok:
    log.warning("Face: placeholder only")
    flux_model['models']['face'] = {
        'base_model': 'insightface/buffalo_l',
        'role': 'face_detection_recognition',
        'lazy_load': True,
        'weights': None,
    }

log.cell_end("InsightFace", "PASS" if face_ok else "PARTIAL")

In [ ]:
"""Cell 7: Embed Object Detection (OWL-ViT2)"""

log.cell_start("Object Detection")
cleanup()

obj_ok = False

# Models to try in order
models_to_try = [
    ('google/owlv2-base-patch16-ensemble', 'Owlv2ForObjectDetection', 'OWL-ViT2'),
    ('google/owlvit-base-patch32', 'OwlViTForObjectDetection', 'OWL-ViT'),
    ('hustvl/yolos-small', 'AutoModelForObjectDetection', 'YOLOS'),
]

for model_id, class_name, display in models_to_try:
    if obj_ok:
        break
    
    try:
        print(f"  Trying {display}...")
        
        if class_name == 'Owlv2ForObjectDetection':
            from transformers import Owlv2ForObjectDetection as ModelClass
        elif class_name == 'OwlViTForObjectDetection':
            from transformers import OwlViTForObjectDetection as ModelClass
        else:
            from transformers import AutoModelForObjectDetection as ModelClass
        
        model = ModelClass.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
        )
        
        weights, params = embed_torch_model(model, 'fp16')
        
        config = {}
        if hasattr(model, 'config'):
            try:
                config = model.config.to_dict()
            except:
                pass
        
        flux_model['models']['object_detect'] = {
            'base_model': model_id,
            'quantization': 'fp16',
            'total_params': params,
            'weights': weights,
            'config': config,
            'role': 'object_detection',
            'tasks': ['detect_objects', 'find_object', 'open_vocab_detection'],
            'lazy_load': True,
        }
        
        print(f"  ✓ {display}: {params:,} params ({params*2/1e9:.2f} GB)")
        del model
        cleanup()
        obj_ok = True
        log.success(f"{display} embedded: {params:,} params")
        
    except Exception as e:
        print(f"  ✗ {display} failed: {e}")

if not obj_ok:
    log.warning("Object detection: placeholder only")
    flux_model['models']['object_detect'] = {
        'base_model': 'google/owlv2-base-patch16-ensemble',
        'role': 'object_detection',
        'lazy_load': True,
        'weights': None,
    }

log.cell_end("Object Detection", "PASS" if obj_ok else "PARTIAL")

In [ ]:
"""Cell 8: Embed Pose Estimation (HRNet)"""

log.cell_start("Pose Estimation")
cleanup()

pose_ok = False

KEYPOINTS = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle'
]

# Models to try
models_to_try = [
    ('hrnet_w32.ms_in1k', 'HRNet-W32'),
    ('hrnet_w18.ms_in1k', 'HRNet-W18'),
    ('vit_base_patch16_224', 'ViT-Base'),
]

for timm_name, display in models_to_try:
    if pose_ok:
        break
    
    try:
        print(f"  Trying {display}...")
        import timm
        
        model = timm.create_model(
            timm_name,
            pretrained=True,
            num_classes=17 * 3,  # 17 keypoints × (x, y, conf)
        )
        model.eval()
        
        weights, params = embed_torch_model(model, 'fp16')
        
        flux_model['models']['pose'] = {
            'base_model': f'timm/{timm_name}',
            'quantization': 'fp16',
            'total_params': params,
            'weights': weights,
            'config': {'num_keypoints': 17, 'keypoints': KEYPOINTS},
            'role': 'pose_estimation',
            'tasks': ['pose_detect', 'gesture', 'body_tracking'],
            'lazy_load': True,
        }
        
        print(f"  ✓ {display}: {params:,} params ({params*2/1e9:.2f} GB)")
        del model
        cleanup()
        pose_ok = True
        log.success(f"{display} embedded: {params:,} params")
        
    except Exception as e:
        print(f"  ✗ {display} failed: {e}")

if not pose_ok:
    log.warning("Pose: placeholder only")
    flux_model['models']['pose'] = {
        'base_model': 'timm/hrnet_w32',
        'role': 'pose_estimation',
        'lazy_load': True,
        'weights': None,
    }

log.cell_end("Pose Estimation", "PASS" if pose_ok else "PARTIAL")

In [ ]:
"""Cell 9: Embed Pyannote (Speaker Diarization)"""

log.cell_start("Pyannote")
cleanup()

speaker_ok = False

if not hf_token:
    print("  ⚠ No HF token - pyannote requires authentication")
    print("  Visit: https://huggingface.co/pyannote/speaker-diarization-3.1")
else:
    try:
        print("  Loading Pyannote models...")
        from pyannote.audio import Model as PyannoteModel
        
        pyannote_state = {'models': {}}
        total_params = 0
        
        # Segmentation
        try:
            print("    Loading segmentation...")
            seg = PyannoteModel.from_pretrained(
                "pyannote/segmentation-3.0",
                use_auth_token=hf_token
            )
            w, p = embed_torch_model(seg, 'fp16')
            pyannote_state['models']['segmentation'] = w
            total_params += p
            print(f"      segmentation: {p:,} params")
            del seg
        except Exception as e:
            print(f"      ✗ segmentation: {e}")
        
        # Embedding
        try:
            print("    Loading embedding...")
            emb = PyannoteModel.from_pretrained(
                "pyannote/embedding",
                use_auth_token=hf_token
            )
            w, p = embed_torch_model(emb, 'fp16')
            pyannote_state['models']['embedding'] = w
            total_params += p
            print(f"      embedding: {p:,} params")
            del emb
        except Exception as e:
            print(f"      ✗ embedding: {e}")
        
        cleanup()
        
        if total_params > 0:
            flux_model['models']['speaker_detect'] = {
                'base_model': 'pyannote/speaker-diarization-3.1',
                'quantization': 'fp16',
                'total_params': total_params,
                'state': pyannote_state,
                'config': {'sample_rate': 16000},
                'role': 'speaker_diarization',
                'tasks': ['speaker_detect', 'diarization', 'voice_activity'],
                'lazy_load': True,
            }
            print(f"  ✓ Pyannote: {total_params:,} params")
            speaker_ok = True
            log.success(f"Pyannote embedded: {total_params:,} params")
            
    except Exception as e:
        print(f"  ✗ Pyannote failed: {e}")

if not speaker_ok:
    log.warning("Speaker: placeholder only")
    flux_model['models']['speaker_detect'] = {
        'base_model': 'pyannote/speaker-diarization-3.1',
        'role': 'speaker_diarization',
        'lazy_load': True,
        'weights': None,
    }

log.cell_end("Pyannote", "PASS" if speaker_ok else "PARTIAL")

In [ ]:
"""Cell 10: Update Orchestration & Metadata"""

log.cell_start("Orchestration")

# Ensure orchestration exists
if 'orchestration' not in flux_model:
    flux_model['orchestration'] = {}
if 'model_triggers' not in flux_model['orchestration']:
    flux_model['orchestration']['model_triggers'] = {}
if 'lazy_models' not in flux_model['orchestration']:
    flux_model['orchestration']['lazy_models'] = []

# Detection triggers
flux_model['orchestration']['model_triggers'].update({
    'face': ['face', 'who is', 'person', 'recognize'],
    'object_detect': ['find', 'locate', 'where is', 'detect'],
    'depth': ['depth', 'distance', 'how far', '3d'],
    'pose': ['pose', 'gesture', 'standing', 'body'],
    'speaker_detect': ['who said', 'speaker', 'voice'],
})

# Camera/audio pipeline
flux_model['orchestration']['camera_active'] = ['face', 'depth']
flux_model['orchestration']['mic_active'] = ['speaker_detect']

flux_model['orchestration']['camera_pipeline'] = {
    'always_on': ['face', 'depth'],
    'on_demand': ['object_detect', 'pose'],
}
flux_model['orchestration']['audio_pipeline'] = {
    'always_on': ['whisper'],
    'on_voice': ['speaker_detect'],
}

# Add to lazy list
for m in ['face', 'object_detect', 'depth', 'pose', 'speaker_detect']:
    if m not in flux_model['orchestration']['lazy_models']:
        flux_model['orchestration']['lazy_models'].append(m)

# Update version
flux_model['version'] = '7.1-detection-embedded'
flux_model['phase'] = 'phase2_5_detection'
flux_model['timestamp'] = datetime.now().isoformat()

# Metadata
if 'metadata' not in flux_model:
    flux_model['metadata'] = {}
flux_model['metadata']['last_modified'] = datetime.now().isoformat()

# Capabilities
new_caps = [
    'detection_stack', 'face_detection', 'object_detection',
    'depth_estimation', 'pose_estimation', 'speaker_diarization',
]
caps = flux_model['metadata'].get('capabilities', [])
for c in new_caps:
    if c not in caps:
        caps.append(c)
flux_model['metadata']['capabilities'] = caps

print(f"  Version: {flux_model['version']}")
print(f"  Capabilities: {len(caps)}")

log.cell_end("Orchestration", "PASS")

In [ ]:
"""Cell 11: Summary & Save"""

log.cell_start("Summary")

print("\n" + "="*60)
print("  DETECTION MODELS SUMMARY")
print("="*60 + "\n")

total_size = 0
embedded = 0
placeholders = 0

for name, state in flux_model.get('models', {}).items():
    params = state.get('total_params', 0)
    size_bytes = state.get('total_size_bytes', 0)
    
    if params > 0:
        size_gb = params * 2 / 1e9
    elif size_bytes > 0:
        size_gb = size_bytes / 1e9
    else:
        size_gb = 0
    
    total_size += size_gb
    
    has_weights = bool(state.get('weights') or state.get('state') or state.get('onnx_models'))
    icon = '✓' if has_weights else '○'
    
    if has_weights:
        embedded += 1
    else:
        placeholders += 1
    
    print(f"  {icon} {name:15} {params:>12,} params  {size_gb:>5.2f} GB")

print(f"\n  Embedded: {embedded} | Placeholders: {placeholders} | Total: {total_size:.2f} GB")
print("="*60)

# Save
OUTPUT_PATH = ROOT / 'checkpoints' / 'Flux-Apex-V1.flx'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"\nSaving to {OUTPUT_PATH}...")
torch.save(flux_model, str(OUTPUT_PATH), pickle_protocol=4)

final_size = OUTPUT_PATH.stat().st_size / 1e9
print(f"✓ Saved: {final_size:.2f} GB")

log.success(f"Model saved: {final_size:.2f} GB")
log.cell_end("Summary", "PASS")

In [ ]:
"""Cell 12: Upload to HuggingFace (Optional)"""

log.cell_start("Upload")

if hf_token:
    from huggingface_hub import HfApi
    
    print("Uploading to HuggingFace Hub...")
    
    try:
        api = HfApi(token=hf_token)
        api.upload_file(
            path_or_fileobj=str(OUTPUT_PATH),
            path_in_repo='checkpoints/Flux-Apex-V1.flx',
            repo_id='UnseenGAP/FLUX',
            commit_message=f'v{flux_model["version"]} - detection models embedded',
        )
        log.success("Uploaded to HuggingFace")
    except Exception as e:
        log.error(f"Upload failed: {e}")
else:
    print("No HF token - skipping upload")

log.cell_end("Upload", "PASS")

In [ ]:
"""Cell 13: Final Report"""

log.separator("PHASE 2.5 COMPLETE")

detection = ['depth', 'face', 'object_detect', 'pose', 'speaker_detect']
det_embedded = sum(1 for d in detection if flux_model['models'].get(d, {}).get('weights') or 
                   flux_model['models'].get(d, {}).get('state') or
                   flux_model['models'].get(d, {}).get('onnx_models'))

print(f"""
╔══════════════════════════════════════════════════════════════╗
║        PHASE 2.5: DETECTION EMBEDDING COMPLETE               ║
╠══════════════════════════════════════════════════════════════╣
║  Output:   {str(OUTPUT_PATH):<47} ║
║  Size:     {final_size:.2f} GB{' ':<45}║
║  Version:  {flux_model['version']:<47} ║
╠══════════════════════════════════════════════════════════════╣
║  Detection Models: {det_embedded}/5 embedded{' ':<35}║
║  Total Models:     {len(flux_model['models']):<42} ║
╠══════════════════════════════════════════════════════════════╣
║  Next: Run phase3_validation.ipynb{' ':<26}║
╚══════════════════════════════════════════════════════════════╝
""")